# Direct Web Research

Call a search function directly, inspect its results, then ask an agent to summarize them.

## 1. Install dependencies

In [ ]:
%pip install -q openai-agents ddgs

## 2. Choose a model provider

Set `PROVIDER` to `"openai"` or `"gemini"`, then add the corresponding API key to Colab Secrets.

In [ ]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "openai"  # Change to "gemini" to use Gemini.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-2.5-flash"

if PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Define and call the search function

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException


def search_web(query: str) -> str:
    """Search the web and return source titles, summaries, and URLs."""
    try:
        results = DDGS(timeout=15).text(query, max_results=5)
    except DDGSException as error:
        print(f"Search failed: {error}")
        return "Search is temporarily unavailable."

    print(f"Search query: {query}")
    print(f"Results found: {len(results)}")
    for index, result in enumerate(results, start=1):
        print(f"\n{index}. {result.get('title', 'Untitled source')}")
        print(result.get('body', 'No summary available.'))
        print(result.get('href', 'No URL available.'))

    return "\n\n".join(
        f"{result.get('title', 'Untitled source')}\n{result.get('body', '')}\n{result.get('href', '')}"
        for result in results
    )


question = "What is agentic AI, and how is it used in business?"
search_results = search_web(question)

## 4. Summarize the search results

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Research Assistant",
    instructions="Summarize only the supplied search results. End with a Sources section containing the URLs you used.",
    model=model,
)

result = await Runner.run(
    starting_agent=agent,
    input=f"Question:\n{question}\n\nSearch results:\n{search_results}",
)

print(result.final_output)